In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/dangkhoa2016/Nodejs-Qdrant-Bilingual-Search.git"
ROOT="/kaggle/working/Nodejs-Qdrant-Bilingual-Search"
if [[ -d "$ROOT/.git" ]]; then
  git -C "$ROOT" fetch origin main
  git -C "$ROOT" reset --hard origin/main
  git -C "$ROOT" clean -ffd
else
  rm -rf "$ROOT"
  git clone --depth 1 "$REPO_URL" "$ROOT"
fi


# Bilingual Semantic Search with Node.js + Qdrant + Qwen3-Embedding-4B — Kaggle CPU-FP16 Production Demo / Demo production CPU-FP16 trên Kaggle

**English**

This notebook runs the canonical bilingual semantic-search stack on a **Kaggle CPU/RAM-only session** using `Qwen/Qwen3-Embedding-4B` in Transformers FP16 mode, the canonical 20K Qdrant snapshot, and the Node.js/Hono REST API.

Before running:
- Enable **Internet**.
- Set **Accelerator=None**.
- Attach model `dangkhoa2016/qwen-qwen3-embedding-4b` → `Transformers/default`.
- Attach dataset `dangkhoa2016/qdrant-bilingual-search-canonical-v2-1-20k`.
- Use **Restart Session → Run All** for a clean, reproducible run.

The notebook has two layers:
1. **Core local demo — required:** Qdrant `127.0.0.1:6333`, embedding service `127.0.0.1:8001`, Node API `127.0.0.1:3000`.
2. **Authenticated public demo — optional:** gateway `127.0.0.1:8090` + Cloudflare Quick Tunnel. It is disabled by default and should only be enabled when you intentionally want to test public access.

**Tiếng Việt**

Notebook này chạy toàn bộ hệ thống semantic search song ngữ chuẩn trên **Kaggle chỉ dùng CPU/RAM** với `Qwen/Qwen3-Embedding-4B` ở chế độ Transformers FP16, snapshot Qdrant 20K chuẩn và REST API Node.js/Hono.

Trước khi chạy:
- Bật **Internet**.
- Chọn **Accelerator=None**.
- Gắn model `dangkhoa2016/qwen-qwen3-embedding-4b` → `Transformers/default`.
- Gắn dataset `dangkhoa2016/qdrant-bilingual-search-canonical-v2-1-20k`.
- Dùng **Restart Session → Run All** để chạy từ trạng thái sạch và có thể tái lập.

Notebook gồm hai lớp:
1. **Core local demo — bắt buộc:** Qdrant `127.0.0.1:6333`, embedding service `127.0.0.1:8001`, Node API `127.0.0.1:3000`.
2. **Authenticated public demo — tùy chọn:** gateway `127.0.0.1:8090` + Cloudflare Quick Tunnel. Mặc định tắt; chỉ bật khi bạn chủ động muốn kiểm thử truy cập công khai.


## 1. Verify the official repository checkout / Xác minh mã nguồn chính thức

**Required / Bắt buộc**

**English:** This section verifies that Kaggle is using the official GitHub repository, that all required helper scripts exist, and that the checkout starts clean. It also shows the exact Git HEAD used by the run.

Configuration:
- `RUN_LIVE_DEMO=True`: execute the real model/Qdrant/API workflow.
- `ENABLE_PUBLIC_TUNNEL=False`: keep Sections 6–7 optional and disabled by default.

The source checkout under `/kaggle/working/Nodejs-Qdrant-Bilingual-Search` is disposable. During hard refresh, stale **untracked** files/directories from earlier Kaggle attempts (for example `snapshots/`) are removed after the reset. Runtime data that must survive the refresh lives outside this source checkout.

Expected result: a clean Git status and an exact HEAD SHA.

**Tiếng Việt:** Phần này xác minh Kaggle đang dùng đúng repository GitHub chính thức, đầy đủ helper script cần thiết và working tree ban đầu sạch. Phần này cũng in chính xác Git HEAD được dùng cho lần chạy.

Cấu hình:
- `RUN_LIVE_DEMO=True`: chạy workflow thật với model/Qdrant/API.
- `ENABLE_PUBLIC_TUNNEL=False`: mặc định không chạy public tunnel; Section 6–7 vẫn là tùy chọn.

Source checkout tại `/kaggle/working/Nodejs-Qdrant-Bilingual-Search` được xem là vùng mã nguồn có thể tạo lại. Khi hard-refresh, các file/thư mục **untracked** còn sót từ lần chạy Kaggle trước (ví dụ `snapshots/`) sẽ được xóa sau khi reset. Runtime data cần giữ lại được đặt bên ngoài source checkout này.

Kết quả mong đợi: Git status sạch và có SHA chính xác của HEAD.


In [ ]:
from pathlib import Path
import os, subprocess, time, urllib.request

WORK = Path("/kaggle/working")
ROOT = WORK / "Nodejs-Qdrant-Bilingual-Search"
RUN_LIVE_DEMO = True
ENABLE_PUBLIC_TUNNEL = False
public_started = False
public_completed = False
evidence_completed = False

required = [
    ROOT / ".git",
    ROOT / "run.sh",
    ROOT / "scripts/kaggle/run-qwen3-transformers-fp16-cpu.sh",
    ROOT / "scripts/kaggle/restore-canonical-qdrant-snapshot.sh",
    ROOT / "scripts/kaggle/production-demo-auth-gateway.mjs",
    ROOT / "scripts/kaggle/start-authenticated-production-demo.sh",
    ROOT / "scripts/kaggle/stop-authenticated-production-demo.sh",
    ROOT / "scripts/kaggle/production-demo-acceptance.mjs",
    ROOT / "scripts/kaggle/collect-production-demo-notebook-evidence.sh",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Incomplete Git checkout:\n- " + "\n- ".join(missing))

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
status = subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True)
if status.strip():
    raise RuntimeError("Refusing to continue from a dirty checkout:\n" + status)
print("ROOT =", ROOT)
print("HEAD =", head)
print("Git status = clean")
print("RUN_LIVE_DEMO =", RUN_LIVE_DEMO)
print("ENABLE_PUBLIC_TUNNEL =", ENABLE_PUBLIC_TUNNEL)


## 2. Validate model input and restore the canonical 20K snapshot / Xác minh model và khôi phục snapshot Qdrant 20K chuẩn

**Required / Bắt buộc**

**English:** This section confirms the CPU-FP16 Qwen3-Embedding-4B runtime contract, resolves the attached model read-only from `/kaggle/input`, verifies the canonical Qdrant snapshot by filename, size and SHA-256, then restores the exact 20K collection. It does **not** reseed or rebuild the dataset. Qdrant runtime snapshots and temporary restore files are intentionally written under `/kaggle/working/qdrant-bilingual-search/snapshot-restore-runtime`, outside the source checkout, so Git must remain clean through later evidence packaging.

Expected result includes `QDRANT_SNAPSHOT_RESTORE=PASS`, `RESEED_PERFORMED=NO`, `20000/20000`, vector size `2560`, distance `Cosine`.

**Tiếng Việt:** Phần này xác nhận contract CPU-FP16 của Qwen3-Embedding-4B, tìm model đã attach trực tiếp từ `/kaggle/input` ở chế độ read-only, kiểm tra snapshot Qdrant chuẩn bằng tên file, kích thước và SHA-256, sau đó restore đúng collection 20K. Phần này **không reseed và không rebuild dataset**. Runtime snapshots và file tạm của Qdrant được chủ động đặt dưới `/kaggle/working/qdrant-bilingual-search/snapshot-restore-runtime`, bên ngoài source checkout, để Git tiếp tục sạch cho đến bước đóng gói evidence.

Kết quả mong đợi gồm `QDRANT_SNAPSHOT_RESTORE=PASS`, `RESEED_PERFORMED=NO`, `20000/20000`, vector size `2560`, distance `Cosine`.


In [ ]:
if RUN_LIVE_DEMO:
    gpu = subprocess.run(["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi -L"], text=True, capture_output=True)
    if gpu.returncode == 0 and gpu.stdout.strip():
        raise RuntimeError("GPU detected. Use Accelerator=None for the canonical CPU-FP16 profile.")
    env = os.environ.copy()
    env["QWEN3_TRANSFORMERS_DRY_RUN"] = "1"
    env["DEMO_PUBLIC"] = "0"
    env["QDRANT_STORAGE_PATH"] = "/kaggle/working/qdrant-bilingual-search/qdrant-data"
    subprocess.run(["bash", "scripts/kaggle/run-qwen3-transformers-fp16-cpu.sh"], cwd=ROOT, env=env, check=True)
    subprocess.run(["bash", "scripts/kaggle/restore-canonical-qdrant-snapshot.sh"], cwd=ROOT, env=env, check=True)
else:
    print("SKIP live input/restore validation")


## 3. Start the canonical local-only stack / Khởi động stack chuẩn chỉ truy cập localhost

**Required / Bắt buộc**

**English:** Starts Qdrant, the Python/FastAPI Qwen3-Embedding-4B embedding service, and the Node.js/Hono API. All three backend services are intentionally bound to loopback addresses. The legacy direct public tunnel is explicitly disabled with `DEMO_PUBLIC=0`.

Expected topology:
`Node API 127.0.0.1:3000 → Embedding 127.0.0.1:8001 + Qdrant 127.0.0.1:6333`.

**Tiếng Việt:** Khởi động Qdrant, embedding service Qwen3-Embedding-4B bằng Python/FastAPI và API Node.js/Hono. Cả ba backend đều cố ý chỉ bind vào loopback. Direct public tunnel kiểu cũ bị tắt rõ ràng bằng `DEMO_PUBLIC=0`.

Topology mong đợi:
`Node API 127.0.0.1:3000 → Embedding 127.0.0.1:8001 + Qdrant 127.0.0.1:6333`.


In [ ]:
if RUN_LIVE_DEMO:
    env = os.environ.copy()
    env.update({
        "DEMO_PUBLIC": "0",
        "QDRANT_VERSION": "1.18.3",
        "QDRANT_STORAGE_PATH": "/kaggle/working/qdrant-bilingual-search/qdrant-data",
    })
    subprocess.run(["bash", "scripts/kaggle/run-qwen3-transformers-fp16-cpu.sh", "start"], cwd=ROOT, env=env, check=True)
else:
    print("SKIP server start")


## 4. Wait for real readiness and inspect safe runtime metadata / Chờ hệ thống sẵn sàng thật và kiểm tra metadata an toàn

**Required / Bắt buộc**

**English:** This is a real readiness gate, not a fixed sleep. It waits until Qdrant, the Qwen3-Embedding-4B embedding service, and the Node API answer successfully, then prints bounded metadata for the collection, embedding model and API configuration. Model startup on CPU can take several minutes.

Expected result: HTTP 200 readiness from ports `6333`, `8001`, and `3000`, with canonical model/collection metadata.

**Tiếng Việt:** Đây là readiness gate thật, không dùng thời gian chờ cố định. Cell sẽ đợi đến khi Qdrant, embedding service Qwen3-Embedding-4B và Node API trả lời thành công, sau đó in metadata giới hạn/an toàn của collection, model embedding và API. Việc nạp model trên CPU có thể mất vài phút.

Kết quả mong đợi: readiness HTTP 200 từ các port `6333`, `8001`, `3000` cùng metadata model/collection đúng chuẩn.


In [ ]:
if RUN_LIVE_DEMO:
    def wait_http(url, timeout=600):
        deadline = time.time() + timeout
        last = None
        while time.time() < deadline:
            try:
                with urllib.request.urlopen(url, timeout=5) as r:
                    body = r.read().decode()
                    if r.status == 200:
                        return body
            except Exception as exc:
                last = exc
            time.sleep(2)
        raise TimeoutError(f"{url} not ready: {last}")

    print(wait_http("http://127.0.0.1:6333/"))
    print(wait_http("http://127.0.0.1:8001/health", timeout=1800))
    print(wait_http("http://127.0.0.1:3000/ready", timeout=300))
    for url in [
        "http://127.0.0.1:6333/collections/knowledge_entities_qwen3_4b_text_v21",
        "http://127.0.0.1:8001/model",
        "http://127.0.0.1:3000/api/v1/info",
    ]:
        print("\n", url)
        print(wait_http(url, timeout=60)[:6000])
else:
    print("SKIP readiness")


## 5. Run stable local semantic acceptance / Chạy kiểm thử semantic ổn định trên localhost

**Required / Bắt buộc**

**English:** Runs the deployment acceptance suite through the local Node API. The hard gates are Thailand EN, Tokyo VI, Beijing VI, plus a Casablanca negative case. Known relation-style diagnostics are intentionally not used as release blockers.

Expected result: `PRODUCTION_DEMO_ACCEPTANCE_PASS=7`.

**Tiếng Việt:** Chạy bộ acceptance test qua Node API local. Các hard gate gồm Thailand EN, Tokyo VI, Beijing VI và negative case Casablanca. Những query relation-style đã biết là diagnostic-only sẽ không được dùng làm release blocker.

Kết quả mong đợi: `PRODUCTION_DEMO_ACCEPTANCE_PASS=7`.


In [ ]:
if RUN_LIVE_DEMO:
    log = WORK / "production-demo-local-acceptance.log"
    with log.open("w") as fh:
        subprocess.run(["node", "scripts/kaggle/production-demo-acceptance.mjs"], cwd=ROOT, check=True, stdout=fh, stderr=subprocess.STDOUT)
    print(log.read_text())
else:
    print("SKIP local acceptance")


## 6. Optional: start the authenticated public gateway and Quick Tunnel / Tùy chọn: khởi động gateway có xác thực và Quick Tunnel

**Optional / Tùy chọn — skipped when `ENABLE_PUBLIC_TUNNEL=False`**

**English:** This section is **not required for the core Kaggle demo**. Enable it only when you want to test a temporary public URL. It creates a private Bearer token, starts the authenticated gateway on `127.0.0.1:8090`, and points Cloudflare Quick Tunnel only to that gateway. The token value is never printed.

Security boundary:
`Internet → Quick Tunnel → 127.0.0.1:8090 gateway → 127.0.0.1:3000 Node API`.
Qdrant and the embedding service are never exposed directly.

**Tiếng Việt:** Phần này **không bắt buộc đối với core Kaggle demo**. Chỉ bật khi bạn muốn kiểm thử một URL công khai tạm thời. Cell sẽ tạo Bearer token riêng, chạy authenticated gateway tại `127.0.0.1:8090` và cho Cloudflare Quick Tunnel trỏ duy nhất vào gateway đó. Giá trị token không bao giờ được in ra.

Ranh giới bảo mật:
`Internet → Quick Tunnel → 127.0.0.1:8090 gateway → 127.0.0.1:3000 Node API`.
Qdrant và embedding service không bao giờ được expose trực tiếp.


In [ ]:
if RUN_LIVE_DEMO and ENABLE_PUBLIC_TUNNEL:
    subprocess.run(["bash", "scripts/kaggle/start-authenticated-production-demo.sh"], cwd=ROOT, check=True)
    public_runtime = ROOT / ".runtime/production-demo-public"
    public_url = (public_runtime / "public.url").read_text().strip()
    token_file = public_runtime / "demo-token"
    print("PUBLIC_URL =", public_url)
    print("TOKEN_FILE =", token_file)
    print("Public API requires Authorization: Bearer <token from TOKEN_FILE>")
    public_started = True
else:
    print("Public tunnel disabled")


## 7. Optional: verify authenticated public acceptance / Tùy chọn: xác minh public acceptance có xác thực

**Optional / Tùy chọn — requires Section 6**

**English:** Re-runs the same stable semantic acceptance through the public URL using the Bearer token file. It also verifies that an unauthenticated public health request receives `401`. A public PASS is recorded only after this cell actually completes successfully.

Expected result when enabled: unauthenticated `401` + authenticated `PRODUCTION_DEMO_ACCEPTANCE_PASS=8`.

**Tiếng Việt:** Chạy lại cùng bộ semantic acceptance ổn định qua public URL bằng Bearer token file. Phần này cũng xác minh request public không có xác thực phải nhận `401`. Public PASS chỉ được ghi nhận khi cell này thực sự chạy xong thành công.

Kết quả mong đợi khi bật: unauthenticated `401` + authenticated `PRODUCTION_DEMO_ACCEPTANCE_PASS=8`.


In [ ]:
if RUN_LIVE_DEMO and ENABLE_PUBLIC_TUNNEL:
    env = os.environ.copy()
    env["API_URL"] = public_url
    env["DEMO_BEARER_TOKEN_FILE"] = str(token_file)
    log = WORK / "production-demo-public-acceptance.log"
    with log.open("w") as fh:
        subprocess.run(["node", "scripts/kaggle/production-demo-acceptance.mjs"], cwd=ROOT, env=env, check=True, stdout=fh, stderr=subprocess.STDOUT)
    print(log.read_text())
    public_completed = True
else:
    print("SKIP public acceptance")


## 8. Collect sanitized, portable evidence / Thu thập evidence đã làm sạch và có thể mang sang máy khác

**Required for release evidence / Bắt buộc nếu dùng làm release evidence**

**English:** Packages runtime identities, endpoint snapshots, acceptance logs, resource usage, listeners and process metadata into a checksummed ZIP. Sensitive command-line arguments and Bearer-token values are excluded. The external `.zip.sha256` uses only the ZIP basename so `sha256sum -c` remains portable after download. The source checkout must still be clean here; evidence packaging is a required part of the overall notebook PASS, not merely an optional post-processing step.

The collector also fails closed if Qdrant, embedding, or Node API listens on a wildcard interface.

**Tiếng Việt:** Đóng gói runtime identity, endpoint snapshot, acceptance log, mức sử dụng tài nguyên, listener và process metadata vào ZIP có checksum. Command-line argument nhạy cảm và giá trị Bearer token bị loại bỏ. File `.zip.sha256` bên ngoài chỉ chứa basename của ZIP để `sha256sum -c` vẫn dùng được sau khi tải về máy khác. Source checkout phải tiếp tục sạch tại đây; bước đóng gói evidence là điều kiện bắt buộc để notebook đạt overall PASS, không chỉ là hậu xử lý tùy chọn.

Collector cũng fail-closed nếu Qdrant, embedding hoặc Node API lắng nghe trên wildcard interface.


In [ ]:
if RUN_LIVE_DEMO:
    subprocess.run(["bash", "scripts/kaggle/collect-production-demo-notebook-evidence.sh"], cwd=ROOT, check=True)
    packages = sorted(WORK.glob("nodejs-qdrant-v1.0.0-production-demo-evidence-*.zip"))
    sidecars = sorted(WORK.glob("nodejs-qdrant-v1.0.0-production-demo-evidence-*.zip.sha256"))
    if not packages or not sidecars:
        raise RuntimeError("Evidence packaging completed without the expected ZIP and portable SHA256 sidecar")
    evidence_completed = True
    print("Latest evidence ZIP =", packages[-1])
    print("Latest sidecar =", sidecars[-1])
else:
    print("SKIP evidence")


## 9. Final status / Trạng thái cuối cùng

**English:** Prints the live lifecycle status and separates the **core local demo**, **required evidence packaging**, and the **optional authenticated public demo**. Public PASS markers are printed only if Sections 6 and 7 actually completed successfully. Overall notebook PASS is printed only when required evidence collection completed; if public mode was intentionally enabled, public acceptance must also complete. Otherwise the overall state is `INCOMPLETE`.

**Tiếng Việt:** In trạng thái lifecycle thực tế và tách riêng **core local demo**, **bước đóng gói evidence bắt buộc**, và **authenticated public demo tùy chọn**. Các marker public PASS chỉ được in nếu Section 6 và 7 thực sự chạy xong thành công. Overall PASS của notebook chỉ xuất hiện sau khi evidence collection bắt buộc hoàn tất; nếu public mode được chủ động bật thì public acceptance cũng phải hoàn tất. Nếu không, trạng thái tổng thể là `INCOMPLETE`.


In [ ]:
if RUN_LIVE_DEMO:
    subprocess.run(["bash", "run.sh", "status"], cwd=ROOT, check=False)
    print("CORE_LOCAL_DEMO=PASS")
    if evidence_completed:
        print("EVIDENCE_COLLECTION=PASS")
    else:
        print("EVIDENCE_COLLECTION=FAIL")
    if ENABLE_PUBLIC_TUNNEL and public_completed:
        print("AUTH_GATEWAY=PASS")
        print("UNAUTHENTICATED_REQUEST=401")
        print("PUBLIC_TUNNEL_TARGET=http://127.0.0.1:8090")
        print("AUTHENTICATED_PUBLIC_DEMO=PASS")
    elif public_started:
        print("AUTHENTICATED_PUBLIC_DEMO=INCOMPLETE")
    else:
        print("AUTHENTICATED_PUBLIC_DEMO=NOT_RUN")
    if evidence_completed and (not ENABLE_PUBLIC_TUNNEL or public_completed):
        print("PRODUCTION_ORIENTED_DEMO_NOTEBOOK=PASS")
    else:
        print("PRODUCTION_ORIENTED_DEMO_NOTEBOOK=INCOMPLETE")
else:
    print("SOURCE_ONLY_NOTEBOOK_VALIDATION=PASS")
